## Topic 7: Incremental Ingestion

### Concept, Intuition, Why It Exists

- The naive approach — "every time we ingest, re-read, re-chunk, and re-embed every single source file" — works fine for a handful of demo files and breaks down hard at real scale: re-embedding thousands of unchanged documents daily wastes compute, money, and time on content that hasn't changed at all since yesterday.
- This is the same discipline `insert_fd_record()` / `update_fd_record()` already established in the FD database layer for the records table: a real database doesn't get dropped and rebuilt from scratch on every update — it tracks individual record changes and only touches what actually changed. Incremental ingestion applies that same principle to **files** instead of **rows**: a manifest remembers what's already been ingested, and only new or changed files get reprocessed.

### Internal Working

1. A manifest is a flat record mapping each tracked file path to the hash of its content and the timestamp it was last ingested.
2. Loading the manifest returns an empty record on the very first run — no manifest exists yet, so nothing has "been seen before."
3. Checking which files need ingestion computes each tracked file's current hash (of the raw file bytes, not just its text content — this hashes the whole file, catching any byte-level change) and compares it against what the manifest has recorded. A mismatch, or no entry at all, means "needs (re)ingesting"; a match means "skip, nothing changed."
4. Updating a manifest entry records the file's current hash and an ingestion timestamp — but only in memory at first.
5. Saving the manifest is the only step that actually writes to disk, and it happens once, after processing — this ordering has real consequences explored below.

### How It's Implemented In This Project

- Demonstrated as a deliberate before/after: a first run against an empty manifest returns every tracked JSON file as needing ingestion; after updating and saving the manifest, a second run against the same unchanged files returns an empty list. Tested directly by changing the content of exactly one tracked file between runs and confirming only that file reappears as needing ingestion — the unchanged files are never touched, which is the entire point, and the part most naive "just reload everything" pipelines get wrong.

### Real-World Issues, Edge Cases, Debugging, Scaling

- **Concurrent writers**: the manifest is a single file. Two ingestion jobs running concurrently and both saving results in last-write-wins — whichever job saves last silently overwrites the other's updates, with no error and no warning. At real scale, the fix is a proper datastore with atomic/transactional writes, not a flat file being read-modified-written by multiple processes.
- **Keyed on path, not content**: if a file gets renamed but its content is byte-identical, it's treated as brand new and fully re-ingested — a false positive, wasted compute on content that hasn't actually changed. Whether this is "wrong" is a genuine trade-off, not an obvious bug to fix: correcting it means also checking new files' hashes against every existing manifest entry's hash, not just the matching path, which adds real complexity for a case that may be rare depending on how often renames actually happen in your real source.
- **Crash recovery**: updating a manifest entry only mutates the in-memory record, file by file, inside a loop; saving runs once, after the loop finishes. A crash midway through processing several changed files means nothing gets persisted: on restart, all of them still look unchanged-since-last-save and correctly get reprocessed. This is safe — no data corruption, no partial/incorrect state — but wasteful, since every file gets redone even if most had already succeeded before the crash. The fix is saving the manifest incrementally, after each file, trading a bit of extra I/O for not having to redo already-completed work after a crash.
- **Embedding model version is not tracked**: the manifest tracks content hash only, not which embedding model produced the resulting vectors in the store. Swap embedding models, and the manifest happily reports "nothing to ingest" while every existing vector is now incomparable to new queries embedded with the new model — a real blind spot, not a minor one. A production manifest should also record the embedding model name/version per entry, and treat a model change as invalidating every existing entry.
- **Deletions are invisible**: if a source file is deleted from disk, the manifest still has a stale entry for it, and nothing in this design removes the corresponding vectors from the store — incremental ingestion as built here only ever adds or updates, never tombstones.

### Design Decisions & Trade-offs

- File-level granularity, not row/page-level: matches the granularity at which sources actually change in this project, but means a single-character change anywhere in a large file forces re-ingesting the entire file, not just the changed page or row. Acceptable when source files are small; would need finer-grained hashing for very large individual files where that waste becomes significant.
- A single flat manifest file vs. a real datastore: simplest possible implementation for a single-process, single-writer context — explicitly not safe for concurrent writers, which is named as a known limitation rather than silently ignored.
- Hashing whole file bytes vs. hashing extracted text content: hashing raw bytes catches any change, including ones that wouldn't affect extracted text. Hashing extracted text instead would be more precise about "did the actual content change" but requires re-parsing the file just to check whether parsing it again is even necessary — a chicken-and-egg cost that whole-file hashing avoids entirely.

### Alternatives & When To Use Each

- A flat manifest file, file-hash keyed — single-writer, small-to-medium file counts, simplicity prioritized.
- A real database table for the manifest — concurrent ingestion jobs, need transactional updates, want to query ingestion history.
- Content-addressable storage, where the hash of a piece of content is its own storage key — unifies deduplication and change-detection into one system, common in data lake architectures.
- Vector-store-native upsert using content hash as part of the vector identity — production RAG systems where the vector store itself should be the single source of truth for "is this already indexed," not a separate manifest file.
- Change Data Capture from a source database — when the source is a live database rather than flat files, streaming row-level changes in near-real-time instead of polling file hashes on a schedule.

### Common Mistakes & Production Failures

- Re-ingesting everything daily "to be safe," defeating the entire purpose of incremental ingestion and quietly burning embedding budget at scale.
- Never saving the manifest incrementally, so any mid-run crash forces a full redo of an entire batch instead of resuming from where it left off.
- Swapping an embedding model without realizing the manifest gives no signal that every existing vector is now stale relative to the new model.
- Treating an empty "files to ingest" result as proof the knowledge base is fully up to date, without separately verifying that file deletions are also being handled — they aren't, in this design.

### Lead-Level Interview Questions

**Q: The manifest is a single file. Two ingestion jobs run concurrently and both try to update it. What happens?**
A: Last write wins — whichever job saves last silently overwrites the other's updates, with no error and no warning. At real scale, the actual fix is a proper datastore with atomic/transactional writes, not a flat file being read-modified-written by multiple processes.

**Q: The manifest keys on file path, not content. A file gets renamed but the content is byte-identical. What happens, and is that the right behavior?**
A: It gets treated as brand new and fully re-ingested — a false positive, wasted compute on unchanged content. Whether that's "wrong" is a genuine trade-off: fixing it means hashing against the manifest's existing content hashes too, not just the matching path, which adds complexity for a case that may be rare in practice. The senior-level answer isn't "obviously fix it" — it's naming the trade-off and deciding based on how often renames actually happen in your real source.

**Q: If ingestion crashes halfway through processing several changed files, what state is the manifest left in, and does a restart resume correctly?**
A: Updating a manifest entry only mutates the in-memory record; saving — the only thing that writes to disk — runs once, after the loop finishes. A crash midway means nothing gets persisted: on restart, all files still look unchanged-since-last-save and correctly get reprocessed. Safe, but wasteful — every file gets redone even if most had already succeeded. The fix is saving incrementally after each file, trading some I/O overhead for not having to redo already-completed work after a crash.

**Q: The manifest tracks content hash only, not which embedding model produced the resulting vectors. If you swap embedding models, does this manifest tell you anything changed?**
A: No — and that's a real blind spot. The manifest would happily report "nothing to ingest" while every existing vector in the store is now incomparable to new queries embedded with the new model. A production manifest needs to track embedding model identity/version per entry and treat a model change as invalidating every existing entry, forcing a full re-embed.

**Q: How does this manifest design handle a source file being deleted?**
A: It doesn't — this is a real gap. The manifest entry for the deleted file simply stays stale forever, and nothing removes the corresponding vectors from the vector store. A complete incremental-ingestion design needs a reconciliation step that compares the manifest's tracked paths against what currently exists on disk and removes vectors for anything no longer present.

### Hidden Concepts Worth Knowing

- **Idempotency**: incremental ingestion is what makes the whole pipeline safely re-runnable — running it twice on unchanged input produces the same end state as running it once, which is a property every production scheduled job should have.
- **Content-addressable storage**: a broader pattern where the hash of a piece of content is its own identifier or storage key — the same hash-as-identity idea this manifest uses for change detection, taken further as the actual storage mechanism.
- **Watermarking / cursor-based incremental processing**: an alternative incremental pattern common in data engineering — track "the last timestamp or offset processed" rather than per-item hashes — works well for append-only logs/streams but doesn't handle in-place file edits the way hash-based tracking does, which is why this project uses hashing instead.

### Revision Summary

> Incremental ingestion mirrors the `insert_fd_record()`/`update_fd_record()` discipline from the FD database, applied to files instead of rows: a manifest tracks each tracked file's content hash; only new-or-changed files get reprocessed; unchanged files are skipped entirely, saving re-chunking, re-embedding, and API cost. Known gaps to name explicitly in any review: not concurrency-safe, keyed on path not content, no embedding-model-version tracking, and no deletion handling.

In [1]:
"""
incremental_ingestion.py
--------------------------
Skip re-embedding files that haven't changed since the last ingestion run --
the same discipline insert_fd_record()/update_fd_record() already established
for the FD database (fd_database.py), applied to source files instead of rows.

A "manifest" (a JSON file) remembers, for every tracked source file:
    { file_path: {"hash": <sha256 of file bytes>, "last_ingested": <UTC ISO timestamp>} }

Each run compares CURRENT file hashes against the manifest. Only NEW or
CHANGED files get (re)ingested -- unchanged files are skipped entirely:
no re-chunking, no re-embedding, no wasted API/embedding cost.

Known limitations (see theory section for the full discussion):
  - Single JSON file -> not safe for concurrent writers (last write wins).
  - Keyed on file PATH, not content -> a rename looks like "new file".
  - Manifest is updated in-memory and only persisted via save_manifest() --
    a crash mid-run loses ALL progress for that run (safe, but wasteful).
  - Does NOT track which embedding model produced existing vectors --
    swapping models silently leaves stale vectors with no warning.
"""

import os
import json
import hashlib
import glob
from datetime import datetime, timezone

MANIFEST_PATH = "ingestion_manifest.json"


def _file_hash(path: str) -> str:
    with open(path, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()


def load_manifest(path: str = MANIFEST_PATH) -> dict:
    """Reads our 'memory' of what's already been ingested. Empty dict on first run."""
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    return {}


def save_manifest(manifest: dict, path: str = MANIFEST_PATH) -> None:
    """Writes the manifest back to disk so the NEXT run can read it."""
    with open(path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)


def get_files_to_ingest(file_paths: list, manifest: dict) -> list:
    """Compares each file's CURRENT hash against the hash recorded last time.
    Different hash (or never seen before) = needs (re)ingesting."""
    to_ingest = []
    for path in file_paths:
        current_hash = _file_hash(path)
        if manifest.get(path, {}).get("hash") != current_hash:
            to_ingest.append(path)
    return to_ingest


def update_manifest_entry(manifest: dict, file_path: str) -> None:
    """Records 'this file, with this exact hash, was ingested just now'.
    Mutates manifest IN MEMORY ONLY -- call save_manifest() to persist."""
    manifest[file_path] = {
        "hash": _file_hash(file_path),
        "last_ingested": datetime.now(timezone.utc).isoformat(),
    }


if __name__ == "__main__":
    JSON_FOLDER = "../data/related_documents_json/"
    tracked_files = sorted(glob.glob(os.path.join(JSON_FOLDER, "*.json")))

    # ---- Run 1: everything is new ----
    manifest = load_manifest()
    to_ingest = get_files_to_ingest(tracked_files, manifest)
    print(f"Run 1 -- files to ingest: {[os.path.basename(f) for f in to_ingest]}")
    for f in to_ingest:
        update_manifest_entry(manifest, f)
    save_manifest(manifest)

    # ---- Run 2: nothing changed, expect empty list ----
    manifest = load_manifest()
    to_ingest = get_files_to_ingest(tracked_files, manifest)
    print(f"Run 2 -- files to ingest: {to_ingest}  (expect empty)")

Run 1 -- files to ingest: []
Run 2 -- files to ingest: []  (expect empty)
